# Synthetic Probe Generation — Method 1: InsightFace Affine Swap

Generates synthetic probes using affine warp + Poisson seamless clone:
- **Source**: random probe from a *different* identity (donor face)
- **Target**: gallery image of the *true* identity (recipient background)
- **Result**: donor face transplanted onto true identity's background context

These probes test whether ArcFace can be fooled by a face that isn't the true identity,
and whether any fooling is driven by genuine face features (Case A) or background/artifacts (Case B).

**Output**:
- Images: `data/synthetic_probes/insightface_swap/{identity}/{identity}_swap_{i}.jpg`
- Metadata: `data/synthetic_probes/metadata.csv` (partial — saliency cols filled later)

**Config**: 50 target identities × 3 probes = 150 synthetic probes

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

SPLIT_PATH      = "splits/lfw_1N_split.json"
GALLERY_EMB     = "cache/G.npy"
GALLERY_IDS     = "cache/gallery_ids.npy"
CRISE_SUMMARY   = "results/crise_maps/summary_crise_tau0.1_N1000_s8_p0.5_MASTERSEED123.csv"

OUT_DIR         = "data/synthetic_probes/insightface_swap"
META_PATH       = "data/synthetic_probes/metadata.csv"
GEN_METHOD      = "insightface_swap"

SYNTH_SEED      = 42
N_TARGET_IDS    = 50
N_PROBES_PER_ID = 3

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import os, json, csv
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from insightface.app import FaceAnalysis

from synth_gen import (
    face_swap_affine,
    get_embedding_from_image,
    select_target_identities,
    select_sources_for_target,
)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(META_PATH), exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# InsightFace setup
# ---------------------------------------------------------------------------

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=(640, 640))
rec = app.models["recognition"]
print("InsightFace ready")

In [ ]:
# ---------------------------------------------------------------------------
# Load split + gallery
# ---------------------------------------------------------------------------

with open(SPLIT_PATH) as f:
    split = json.load(f)

G           = np.load(GALLERY_EMB).astype(np.float32)          # (1680, 512)
gallery_ids = np.load(GALLERY_IDS, allow_pickle=True).tolist()  # list[str]
id_to_index = {gid: i for i, gid in enumerate(gallery_ids)}

print(f"Gallery: {G.shape[0]} identities, {G.shape[1]}-dim embeddings")
print(f"Probe identities in split: {len(split['probes'])}")

In [ ]:
# ---------------------------------------------------------------------------
# Select 50 target identities from identities with completed RISE maps
# (using RISE summary as proxy — all have valid probe images + face detection)
# ---------------------------------------------------------------------------

rise_summary = pd.read_csv(CRISE_SUMMARY)

# Identify the true_id column name (may vary slightly between runs)
id_col = "true_id" if "true_id" in rise_summary.columns else rise_summary.columns[0]
completed_ids = rise_summary[id_col].dropna().unique().tolist()
print(f"Completed RISE maps: {len(completed_ids)} identities")

target_ids = select_target_identities(completed_ids, n=N_TARGET_IDS, seed=SYNTH_SEED)
print(f"Selected {len(target_ids)} target identities (seed={SYNTH_SEED})")
print("First 5:", target_ids[:5])

In [ ]:
# ---------------------------------------------------------------------------
# Main generation loop
# ---------------------------------------------------------------------------

records = []
n_ok = 0
n_fail = 0

for exp_i, target_id in enumerate(tqdm(target_ids, desc="Target identities")):
    target_gallery_path = split["gallery"][target_id]
    target_img = cv2.imread(target_gallery_path)

    if target_img is None:
        print(f"[WARN] Could not read gallery image for {target_id}")
        continue

    id_out_dir = os.path.join(OUT_DIR, target_id)
    os.makedirs(id_out_dir, exist_ok=True)

    source_pairs = select_sources_for_target(
        target_id, exp_i, split, n=N_PROBES_PER_ID, seed=SYNTH_SEED
    )

    for k, (src_id, src_path) in enumerate(source_pairs):
        source_img = cv2.imread(src_path)
        if source_img is None:
            n_fail += 1
            records.append(dict(
                identity=target_id, generation_method=GEN_METHOD,
                source_identity=src_id, source_path=src_path, blend_alpha=None,
                output_path=None, embedding_ok=False,
                arcface_similarity=None, rank1_match=None,
                saliency_cosine_sim=None, saliency_l1=None, case_label=None,
            ))
            continue

        swapped = face_swap_affine(source_img, target_img, app)

        if swapped is None:
            n_fail += 1
            records.append(dict(
                identity=target_id, generation_method=GEN_METHOD,
                source_identity=src_id, source_path=src_path, blend_alpha=None,
                output_path=None, embedding_ok=False,
                arcface_similarity=None, rank1_match=None,
                saliency_cosine_sim=None, saliency_l1=None, case_label=None,
            ))
            continue

        out_fname = f"{target_id}_swap_{k}.jpg"
        out_path = os.path.join(id_out_dir, out_fname)
        cv2.imwrite(out_path, swapped, [cv2.IMWRITE_JPEG_QUALITY, 95])

        n_ok += 1
        records.append(dict(
            identity=target_id, generation_method=GEN_METHOD,
            source_identity=src_id, source_path=src_path, blend_alpha=None,
            output_path=out_path, embedding_ok=True,
            arcface_similarity=None, rank1_match=None,
            saliency_cosine_sim=None, saliency_l1=None, case_label=None,
        ))

print(f"\nGeneration complete: {n_ok} saved, {n_fail} failed")

In [ ]:
# ---------------------------------------------------------------------------
# Visualisation: 6 examples — Source | Swap result | Target (gallery)
# ---------------------------------------------------------------------------

ok_records = [r for r in records if r["output_path"] is not None]
show_targets = target_ids[:6]

fig, axes = plt.subplots(len(show_targets), 3,
                         figsize=(9, 3 * len(show_targets)))

for row, tid in enumerate(show_targets):
    rec_row = next((r for r in ok_records if r["identity"] == tid), None)
    if rec_row is None:
        for col in range(3):
            axes[row, col].axis("off")
        continue

    src_img  = cv2.cvtColor(cv2.imread(rec_row["source_path"]), cv2.COLOR_BGR2RGB)
    tgt_img  = cv2.cvtColor(cv2.imread(split["gallery"][tid]),  cv2.COLOR_BGR2RGB)
    swap_img = cv2.cvtColor(cv2.imread(rec_row["output_path"]), cv2.COLOR_BGR2RGB)

    for col, (img, title) in enumerate([
        (src_img,  f"Source\n{rec_row['source_identity'][:22]}"),
        (swap_img, "Swap result"),
        (tgt_img,  f"Target (gallery)\n{tid[:22]}"),
    ]):
        axes[row, col].imshow(img)
        axes[row, col].set_title(title, fontsize=8)
        axes[row, col].axis("off")

plt.suptitle("InsightFace Affine Swap — Source | Result | Target", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "sample_grid.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Compute ArcFace embeddings for all successfully saved synthetic probes
# ---------------------------------------------------------------------------

synth_embeddings = {}  # output_path -> (512,) embedding or None

for r in tqdm(ok_records, desc="Embedding synthetic probes"):
    emb = get_embedding_from_image(r["output_path"], app, rec)
    synth_embeddings[r["output_path"]] = emb
    if emb is None:
        r["embedding_ok"] = False

n_emb_ok   = sum(1 for e in synth_embeddings.values() if e is not None)
n_emb_fail = sum(1 for e in synth_embeddings.values() if e is None)
print(f"Embeddings: {n_emb_ok} OK, {n_emb_fail} failed (no face detected)")

In [ ]:
# ---------------------------------------------------------------------------
# Compute rank-1 match + cosine similarity to true identity
# ---------------------------------------------------------------------------

for r in records:
    if not r["embedding_ok"] or r["output_path"] is None:
        continue

    emb = synth_embeddings.get(r["output_path"])
    if emb is None:
        r["embedding_ok"] = False
        continue

    sims = G @ emb                                      # (1680,)
    true_idx = id_to_index[r["identity"]]
    sim_true = float(sims[true_idx])
    rank1_idx = int(np.argmax(sims))

    r["arcface_similarity"] = round(sim_true, 6)
    r["rank1_match"]        = (rank1_idx == true_idx)

# Summary
eval_rows = [r for r in records if r["rank1_match"] is not None]
rank1_rate = sum(r["rank1_match"] for r in eval_rows) / max(len(eval_rows), 1)
mean_sim   = np.mean([r["arcface_similarity"] for r in eval_rows])
print(f"Evaluable probes: {len(eval_rows)}")
print(f"Rank-1 match rate: {rank1_rate:.4f}  (fraction fooling ArcFace)")
print(f"Mean cosine sim to true identity: {mean_sim:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Save metadata.csv
# source_path is internal; saliency_cosine_sim, saliency_l1, case_label
# will be filled in after CRISE runs on the synthetic probes.
# ---------------------------------------------------------------------------

META_COLS = [
    "identity", "generation_method", "source_identity", "blend_alpha",
    "output_path", "embedding_ok", "arcface_similarity", "rank1_match",
    "saliency_cosine_sim", "saliency_l1", "case_label",
]

meta_df = pd.DataFrame(records)[META_COLS]
meta_df.to_csv(META_PATH, index=False)
print(f"Saved {len(meta_df)} rows → {META_PATH}")
meta_df.head(10)

In [ ]:
# ---------------------------------------------------------------------------
# Quick stats
# ---------------------------------------------------------------------------

print("=== Generation summary ===")
print(meta_df.groupby("generation_method")[["embedding_ok", "rank1_match"]].agg([
    lambda x: x.sum(), lambda x: x.mean()
]).to_string())

if len(eval_rows) > 0:
    sims_all = [r["arcface_similarity"] for r in eval_rows]
    print(f"\nSimilarity to true identity:")
    print(f"  mean={np.mean(sims_all):.4f}  std={np.std(sims_all):.4f}")
    print(f"  min={np.min(sims_all):.4f}   max={np.max(sims_all):.4f}")
    print(f"  > 0.3 (high sim): {sum(s > 0.3 for s in sims_all)} / {len(sims_all)}")